# F2: signed H/S/P matrices (upcoming)

**Finding** (`notes/theory_audit.md`, F2): crawfish builds its h/p/s "matrices" from $(\mathrm{Re}[T^*_uT_v])^2$.
That makes S and P non-negative and H non-positive. Whenever the true matrix element is positive, the
bonding/antibonding call is wrong: in N2, px–px σ comes out antibonding.

**Target:** section 2.3 of the notes, implemented in `fixcompare.Reference`. It uses complex per-k matrices
$H_t = T\varepsilon T^\dagger$ and $S_t = TT^\dagger$, coefficients from band-space Löwdin (or $S^{-1}T$), and
weights $w_t\,\mathrm{Re}[C^*_u M_{uv} C_v]$.

crawfish's default `state_sep=False` uses the same-cell (R=0) element of each matrix, so the reference is evaluated
the same way (`images="R0"`). For N2, an isolated molecule, this is within 1e-4 Ha of the sum over all images
(`state_sep=True`, `images="all"`).

**This notebook adapts to the implementation status:**
- Before the fix: current crawfish vs. the reference.
- After the fix, which is detected through the legacy switch in `fixcompare.LEGACY_FLAGS["F2"]`: legacy matrices
  vs. fixed crawfish vs. the reference. The tests then require fixed crawfish to match the reference.

If the implementation's default basis differs from the reference's `basis="lowdin"` (open question Q1), change
`REF_BASIS` below.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))  # fixcompare.py lives next to this notebook (run with this directory as cwd)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fixcompare import *
from crawfish.funcs.general import SIGMA_DEFAULT
from crawfish.funcs import get_pcohp, get_ipcohp, get_ipcobi

F2_DONE = fix_available("F2")
REF_BASIS = "lowdin"
if F2_DONE:
    # Isolate the F2 step: later fixes are held at their legacy behaviour in both
    versions = {"legacy matrices": load_edata_at("F2", fixed=False), "fixed": load_edata_at("F2", fixed=True)}
else:
    versions = {"current crawfish": load_edata()}
edata = list(versions.values())[-1]
ref = Reference(edata, basis=REF_BASIS)
A = lambda orbs: orb_idcs(edata, 0, orbs)
B = lambda orbs: orb_idcs(edata, 1, orbs)
PAIRS = [("s", "s"), ("s", "px"), ("px", "px"), ("py", "py"), ("pz", "pz"), (None, None)]
label = lambda o1, o2: "all-all" if o1 is None else f"{o1}-{o2}"
SIG = SIGMA_DEFAULT
erange = np.arange(edata.e_tj.min() - 0.03, edata.e_tj.max() + 0.03, 0.0005, dtype=np.float32)
x_ev = to_ev_rel_mu(edata, erange)
print("F2 implemented:", F2_DONE, "| compared versions:", list(versions))

## −pCOHP spectra (bonding positive)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True)
for ax, (o1, o2) in zip(axes.ravel(), PAIRS):
    for name, ed in versions.items():
        _, spec = get_pcohp(ed, idcs1=0, idcs2=1, orbs1=o1, orbs2=o2, erange=erange, sig=SIG,
                            use_cache=False, use_cached_spectrum=False, save_spectrum=False)
        ax.plot(x_ev, -np.asarray(spec), label=name)
    ax.plot(x_ev, -reference_spectrum(edata, ref.weights_tj(A(o1), B(o2), "cohp", "R0"), erange, SIG),
            "k--", lw=1, label="reference")
    ax.axvline(0, color="gray", lw=0.5)
    ax.set_title(f"N1 – N2: {label(o1, o2)}")
for ax in axes[1]:
    ax.set_xlabel(r"$E - \mu$ (eV)")
for ax in axes[:, 0]:
    ax.set_ylabel("−pCOHP")
axes[0, 0].legend()
fig.tight_layout()
savefig(fig, "F2_pcohp_spectra")
plt.show()

## Integrated pCOHP and pCOBI

In [ ]:
rows = []
for o1, o2 in PAIRS:
    kw = dict(idcs1=0, idcs2=1, orbs1=o1, orbs2=o2, use_cache=False)
    row = {"pair": label(o1, o2)}
    for name, ed in versions.items():
        row[f"IpCOHP {name}"] = last_value(get_ipcohp(ed, **kw))
        row[f"IpCOBI {name}"] = last_value(get_ipcobi(ed, **kw))
    row["IpCOHP reference"] = ref.integrated(A(o1), B(o2), "cohp", "R0")
    row["IpCOBI reference"] = ref.integrated(A(o1), B(o2), "cobi", "R0")
    rows.append(row)
table = pd.DataFrame(rows).set_index("pair")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
table[[c for c in table if c.startswith("IpCOHP")]].plot.bar(ax=axes[0], rot=0)
table[[c for c in table if c.startswith("IpCOBI")]].plot.bar(ax=axes[1], rot=0)
axes[0].set_ylabel("IpCOHP (Ha), negative = bonding")
axes[1].set_ylabel("IpCOBI (provisional, see note)")
for ax in axes:
    ax.axhline(0, color="black", lw=0.5)
fig.tight_layout()
savefig(fig, "F2_integrated")
plt.show()
table.round(5)

The reference IpCOBI integrates $f\,\mathrm{Re}[C^*_u D_{uv} C_v]$, with $D$ the density matrix built from
fillings in [0, 1]. The spin factor and sign convention of COBI still need to be checked against Müller et al.
2021 (open question Q3), so treat the COBI column as provisional.

## Band-energy sum rule (reference)

Summing the integrated COHP over all orbital pairs, including on-site terms, should reproduce the band energy
$\sum w f \varepsilon$, up to the spilling. Below, both are accumulated in order of increasing band energy.

In [ ]:
w, f, e = ref.w_t, ref.f_tj, ref.e_tj
per_state = np.einsum("t,tju,tuv,tjv->tj", w, ref.C.conj(), ref.H, ref.C).real * f
order = np.argsort(e.ravel())
fig, ax = plt.subplots(figsize=(6, 3.5))
x = (e.ravel()[order] - edata.mu) * 27.211386
ax.step(x, np.cumsum(per_state.ravel()[order]), where="post", label=r"$\sum_{uv}$ ICOHP$_{uv}$ (reference)")
ax.step(x, np.cumsum((w[:, None] * f * e).ravel()[order]), where="post", linestyle="--", label=r"$\sum w f \varepsilon$")
ax.set_xlabel(r"$E - \mu$ (eV)")
ax.set_ylabel("cumulative (Ha)")
ax.legend()
savefig(fig, "F2_sum_rule")
plt.show()
print(f"sum rule: {ref.sum_rule():.5f} Ha vs band energy {ref.band_energy():.5f} Ha "
      f"({ref.sum_rule() / ref.band_energy():.2%})")

## Basis and energy-zero dependence of the reference (input for Q1)

In the Löwdin basis, off-diagonal H doesn't depend on the energy zero. In the non-orthogonal basis, shifting ε by
μ changes the ICOHP by −μ·ICOOP (Glassey & Hoffmann 2000). This data uses ortho-orbital projections, so the
non-orthogonal overlap comes only from band truncation.

In [ ]:
variants = {
    "Löwdin": Reference(edata, basis="lowdin"),
    "Löwdin, ε−μ": Reference(edata, basis="lowdin", energy_shift=edata.mu),
    "non-orthogonal": Reference(edata, basis="nonorthogonal"),
    "non-orthogonal, ε−μ": Reference(edata, basis="nonorthogonal", energy_shift=edata.mu),
}
basis_table = pd.DataFrame({name: {label(o1, o2): r.integrated(A(o1), B(o2), "cohp") for o1, o2 in PAIRS}
                            for name, r in variants.items()})
ax = basis_table.plot.bar(rot=0, figsize=(10, 3.5))
ax.axhline(0, color="black", lw=0.5)
ax.set_ylabel("reference IpCOHP (Ha)")
savefig(ax.figure, "F2_basis_energy_zero")
plt.show()
basis_table.round(5)

## Tests

In [ ]:
# Reference sanity: sum rule within 1%, and every N-N pair is net bonding
assert abs(ref.sum_rule() / ref.band_energy() - 1) < 0.01
for o1, o2 in PAIRS:
    assert table.loc[label(o1, o2), "IpCOHP reference"] < 0, (o1, o2)
# Löwdin basis: off-diagonal COHP is independent of the energy zero
assert np.allclose(basis_table["Löwdin"], basis_table["Löwdin, ε−μ"], atol=1e-6)

if F2_DONE:
    from crawfish.funcs.pcohp import _get_pcohp_tj
    fixed = versions["fixed"]
    for o1, o2 in PAIRS:
        for kind in ["IpCOHP", "IpCOBI"]:
            assert np.isclose(table.loc[label(o1, o2), f"{kind} fixed"], table.loc[label(o1, o2), f"{kind} reference"],
                              rtol=1e-3, atol=5e-5), (kind, o1, o2)
        # state_sep=True (all lattice images) against the reference's state-resolved form
        w_all = _get_pcohp_tj(fixed, 0, None, o1, 1, None, o2, use_cache=False, state_sep=True)
        assert np.isclose(np.sum(w_all * fixed.occ_tj), ref.integrated(A(o1), B(o2), "cohp", "all"), atol=2e-5)
    # Legacy matrices are still reachable and still give px-px antibonding
    assert table.loc["px-px", "IpCOHP legacy matrices"] > 0
    print("F2 implemented: fixed crawfish matches the reference (R=0 and all-image forms).")
else:
    # Known-bug check documenting the current state. It isn't reached once F2 lands.
    assert table.loc["px-px", "IpCOHP current crawfish"] > 0, "px-px no longer antibonding: has F2 landed?"
    print("F2 not implemented yet: current crawfish px-px IpCOHP is "
          f"{table.loc['px-px', 'IpCOHP current crawfish']:+.4f} (antibonding), "
          f"reference {table.loc['px-px', 'IpCOHP reference']:+.4f}.")
print("All F2 checks passed.")